# Task 3 — Consultas Athena e dashboard

Pré-requisitos: ETL da Task 2 com status `SUCCEEDED` e Parquet em `s3://<bucket>/curated/`.

Antes de rodar este notebook, execute uma vez:

```bash
python scripts/register_glue_tables.py
```

In [1]:
import sys
from pathlib import Path

import awswrangler as wr
import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import AWS_REGION, GLUE_DATABASE

sns.set_theme(style="whitegrid")
print(f"Região: {AWS_REGION}")
print(f"GLUE_DATABASE = {GLUE_DATABASE!r}")

Região: us-east-1
GLUE_DATABASE = 'grupo5_task2_dw'


## 4.2 — Exploração em `dim_products`

In [2]:
sql_dim_products = """
SELECT
    product_id,
    product_name,
    product_line,
    product_vendor
FROM dim_products
LIMIT 20
"""

df_products = wr.athena.read_sql_query(
    sql=sql_dim_products,
    database=GLUE_DATABASE,
    ctas_approach=False,
)
display(df_products.head())

C:\venv\fgv-task3\Lib\site-packages\awswrangler\athena\_read.py:602: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,product_id,product_name,product_line,product_vendor
0,S10_1678,1969 Harley Davidson Ultimate Chopper,Motorcycles,Min Lin Diecast
1,S10_1949,1952 Alpine Renault 1300,Classic Cars,Classic Metal Creations
2,S10_2016,1996 Moto Guzzi 1100i,Motorcycles,Highway 66 Mini Classics
3,S10_4698,2003 Harley-Davidson Eagle Drag Bike,Motorcycles,Red Start Diecast
4,S10_4757,1972 Alfa Romeo GTA,Classic Cars,Motor City Art Classics


## 4.3 — Vendas totais por país

In [3]:
sql_sales_by_country = """
SELECT
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
GROUP BY dim_countries.country
ORDER BY total_sales DESC
LIMIT 10
"""

df_sales_country = wr.athena.read_sql_query(
    sql=sql_sales_by_country,
    database=GLUE_DATABASE,
    ctas_approach=False,
)
display(df_sales_country)

C:\venv\fgv-task3\Lib\site-packages\awswrangler\athena\_read.py:602: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,country,total_sales
0,USA,3273280.05
1,Spain,1099389.09
2,France,1007374.02
3,Australia,562582.59
4,New Zealand,476847.01
5,UK,436947.44
6,Italy,360616.81
7,Finland,295149.35
8,Singapore,263997.78
9,Denmark,218994.92


## 4.4 — Base analítica (data, linha, produto, país)

In [4]:
sql_detail = """
SELECT
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country,
    SUM(fact_orders.sales_amount) AS total_sales
FROM fact_orders
JOIN dim_products ON fact_orders.product_id = dim_products.product_id
JOIN dim_countries ON fact_orders.country_key = dim_countries.country_key
JOIN dim_dates ON fact_orders.order_date_key = dim_dates.date_key
GROUP BY
    dim_dates.full_date,
    dim_products.product_line,
    dim_products.product_name,
    dim_countries.country
"""

df_detail = wr.athena.read_sql_query(
    sql=sql_detail,
    database=GLUE_DATABASE,
    ctas_approach=False,
)

df_detail["full_date"] = pd.to_datetime(df_detail["full_date"])
display(df_detail.head())
print(f"Linhas: {len(df_detail):,}")

C:\venv\fgv-task3\Lib\site-packages\awswrangler\athena\_read.py:602: UserWarning: No `s3_output` was provided and the workgroup has no ResultConfiguration set. Falling back to the default bucket `aws-athena-query-results-{account}-{region}`. Because S3 bucket names are global, relying on this predictable default is discouraged: pass an explicit `s3_output`, or configure a workgroup with EnforceWorkGroupConfiguration=true and a ResultConfiguration.
  s3_output = _get_s3_output(s3_output=s3_output, wg_config=wg_config, boto3_session=boto3_session)


,full_date,product_line,product_name,country,total_sales
0,2003-01-29,Vintage Cars,1913 Ford Model T Speedster,Norway,2489.13
1,2003-01-29,Trucks and Buses,1996 Peterbilt 379 Stake Bed with Outrigger,Norway,2850.75
2,2003-02-11,Ships,18th century schooner,Denmark,5072.71
3,2003-02-11,Vintage Cars,1912 Ford Model T Delivery Wagon,Denmark,3232.24
4,2003-02-11,Ships,The USS Constitution Ship,Denmark,1882.32


Linhas: 2,996


## 4.5 — Dashboard interativo

In [5]:
min_date = df_detail["full_date"].min().date()
max_date = df_detail["full_date"].max().date()
countries = sorted(df_detail["country"].dropna().unique().tolist())
product_lines = sorted(df_detail["product_line"].dropna().unique().tolist())

date_start = widgets.DatePicker(description="Data início", value=min_date)
date_end = widgets.DatePicker(description="Data fim", value=max_date)
country_filter = widgets.Dropdown(
    options=[("Todos", "ALL")] + [(c, c) for c in countries],
    description="País",
    value="ALL",
)
line_filter = widgets.Dropdown(
    options=[("Todas", "ALL")] + [(p, p) for p in product_lines],
    description="Linha",
    value="ALL",
)
top_n = widgets.IntSlider(value=5, min=1, max=10, step=1, description="Top N")
run_btn = widgets.Button(description="Atualizar gráfico", button_style="primary")
out = widgets.Output()


def filtered_detail() -> pd.DataFrame:
    df = df_detail.copy()
    start = pd.Timestamp(date_start.value)
    end = pd.Timestamp(date_end.value)
    df = df[(df["full_date"] >= start) & (df["full_date"] <= end)]
    if country_filter.value != "ALL":
        df = df[df["country"] == country_filter.value]
    if line_filter.value != "ALL":
        df = df[df["product_line"] == line_filter.value]
    return df


def plot_top_products(_=None) -> None:
    with out:
        out.clear_output(wait=True)
        df = filtered_detail()
        if df.empty:
            print("Nenhum dado para os filtros selecionados.")
            return
        ranking = (
            df.groupby("product_name", as_index=False)["total_sales"]
            .sum()
            .sort_values("total_sales", ascending=False)
            .head(top_n.value)
        )
        plt.figure(figsize=(10, max(4, 0.45 * len(ranking))))
        sns.barplot(
            data=ranking,
            y="product_name",
            x="total_sales",
            hue="product_name",
            dodge=False,
            legend=False,
        )
        plt.xlabel("Vendas totais")
        plt.ylabel("Produto")
        plt.title(f"Top {top_n.value} produtos")
        plt.tight_layout()
        plt.show()


run_btn.on_click(plot_top_products)

display(
    widgets.VBox(
        [
            widgets.HBox([date_start, date_end]),
            widgets.HBox([country_filter, line_filter, top_n]),
            run_btn,
            out,
        ]
    )
)
plot_top_products()